# Principal Component Analysis (PCA) on the Iris Dataset

In this notebook, I use Principal Component Analysis (PCA) to reduce the Iris dataset from four numerical features to two principal components. I standardize the features first, calculate the explained variance ratio, and visualize the samples in the reduced two-dimensional space.

## 1. Import libraries and load the dataset

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import joblib
import pickle

ROOT = Path("..")
IMAGES_DIR = ROOT / "images"
MODELS_DIR = ROOT / "models"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

iris = load_iris(as_frame=True)
X = iris.data.copy()
y = iris.target.copy()

print("Original shape:", X.shape)
display(X.head())

## 2. Standardize the features

I standardize the four input features because PCA is sensitive to feature scales. Standardization helps each feature contribute on a comparable scale.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled shape:", X_scaled.shape)
print("Scaled feature means:", X_scaled.mean(axis=0).round(3))
print("Scaled feature standard deviations:", X_scaled.std(axis=0).round(3))

## 3. Apply PCA to reduce the dataset to 2D

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["Principal Component 1", "Principal Component 2"])
pca_df["species"] = y.map(dict(enumerate(iris.target_names)))
display(pca_df.head())

print("Reduced shape:", X_pca.shape)

## 4. Find the explained variance ratio

The explained variance ratio indicates the fraction of the dataset's variance captured by each principal component. The sum for the first two components tells me how much variance is retained in the 2D representation.

In [ ]:
variance_ratios = pca.explained_variance_ratio_
for index, ratio in enumerate(variance_ratios, start=1):
    print(f"Principal Component {index}: {ratio:.4f} ({ratio * 100:.2f}%)")

total_variance = variance_ratios.sum()
print(f"Total variance explained by 2 components: {total_variance:.4f} ({total_variance * 100:.2f}%)")

variance_df = pd.DataFrame({
    "Principal Component": ["PC1", "PC2"],
    "Explained Variance Ratio": variance_ratios,
    "Explained Variance (%)": variance_ratios * 100
})
display(variance_df.round(4))

## 5. Visualize the two principal components

In [ ]:
plt.figure(figsize=(8, 6))
for species_id, species_name in enumerate(iris.target_names):
    mask = y.to_numpy() == species_id
    plt.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=species_name, s=55, edgecolor="black", linewidth=0.3
    )

plt.title("Iris Dataset Reduced to Two Dimensions Using PCA")
plt.xlabel(f"Principal Component 1 ({variance_ratios[0] * 100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({variance_ratios[1] * 100:.1f}% variance)")
plt.legend(title="True species")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "04_iris_pca_2d.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Plot explained variance

In [ ]:
plt.figure(figsize=(7, 5))
plt.bar(["PC1", "PC2"], variance_ratios)
plt.title("Explained Variance Ratio of the First Two Components")
plt.ylabel("Explained variance ratio")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "05_pca_explained_variance.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Save and reload the PCA pipeline components

I save the fitted scaler and PCA object so that future data can be transformed using the same learned parameters.

In [ ]:
joblib.dump(scaler, MODELS_DIR / "iris_pca_scaler.joblib")
joblib.dump(pca, MODELS_DIR / "iris_pca.joblib")

with open(MODELS_DIR / "iris_pca.pkl", "wb") as file:
    pickle.dump(pca, file)

loaded_scaler = joblib.load(MODELS_DIR / "iris_pca_scaler.joblib")
loaded_pca = joblib.load(MODELS_DIR / "iris_pca.joblib")
sample_2d = loaded_pca.transform(loaded_scaler.transform(X.iloc[:5]))

with open(MODELS_DIR / "iris_pca.pkl", "rb") as file:
    pickled_pca = pickle.load(file)

print("First 5 transformed samples from reloaded model:")
display(pd.DataFrame(sample_2d, columns=["PC1", "PC2"]))
print("joblib and original PCA outputs match:", np.allclose(sample_2d, X_pca[:5]))
print("Pickle model output shape:", pickled_pca.transform(X_scaled[:5]).shape)

## 8. My conclusion

I used PCA to convert the four standardized Iris features into two principal components. The explained variance ratios show how much information each component retains, while the scatter plot helps me inspect how the three known species appear in the reduced space. PCA reduces the number of dimensions, which can make visualization and further analysis easier, although some information is lost when fewer components are retained.